# GLOSARIO

In [1]:
import pandas as pd
import numpy as np
import polars as pl

In [2]:
df_customers = pd.read_csv('../data/sales_customers.csv')
df_employees = pd.read_csv('../data/sales_employees.csv')
df_orders = pd.read_csv('../data/sales_orders.csv')
df_orderarchive = pd.read_csv('../data/sales_ordersarchive.csv')
df_products = pd.read_csv('../data/sales_products.csv')

pl_customers = pl.read_csv('../data/sales_customers.csv')
pl_employees = pl.read_csv('../data/sales_employees.csv')
pl_orders = pl.read_csv('../data/sales_orders.csv')
pl_orderarchive = pl.read_csv('../data/sales_ordersarchive.csv')
pl_products = pl.read_csv('../data/sales_products.csv')

# AS / rename / alias

```SQL
SELECT
    product AS Nombre_Producto,
    price AS Precio_Unitario
FROM sales.products;
````

In [4]:
df_p = df_products.copy()

res = df_p[['product','price']].rename(
    columns={'product':'Nombre_Producto', 'price':'Precio_Unitario'}
)

In [6]:
pl_p = pl_products

res_pl = pl_p.select([
    pl.col('product').alias('Nombre_Producto'),
    pl.col('price').alias('Precio_Unitario')
])

# DISTINCT / drop_duplicates() / unique()

```SQL
SELECT
    DISTINCT orderstatus
FROM sales.orders
```

In [9]:
df_o = df_orders.copy()

res_df = df_o[['orderstatus']].drop_duplicates()

In [10]:
pl_o = pl_orders

res_pl = pl_o.select(pl.col('orderstatus')).unique()

# Limit / head / head

```SQL
SELECT
    orderstatus
FROM sales.orders
LIMIT 2;
```

In [17]:
df_or = df_orders.copy()

res_df = df_or[['orderstatus']].head(2)

In [18]:
pl_or = pl_orderarchive

res_pl = pl_or.select(pl.col('orderstatus')).head(2)

# WHERE + LOWER

```SQL
SELECT
    country
FROM sales.customers
WHERE LOWER(country) = 'germany';
```

In [25]:
df_c = df_customers.copy()

res_df = df_c[
    (df_c['country'].str.lower() == 'germany')
]

res_df = res_df[['country']]

In [26]:
pl_c = pl_customers

res_pl = pl_c.filter(
    (pl.col('country').str.to_lowercase() == 'germany')
).select([
    pl.col('country')
])

# WHERE + Number

```SQL
SELECT
    score
FROM sales.customers
WHERE score > 500;
```

In [48]:
df_o = df_customers.copy()

res_df = df_o[
    (df_o['score'] > 500)
]

res_df = res_df[['score']]

In [47]:
pl_c = pl_customers

res_pl = pl_c.filter(
    (pl.col('score') > 500)
).select(
    pl.col('score')
)

# TRIM + LOWER + IN

```SQL
SELECT
    country
FROM sales.customers
WHERE TRIM(LOWER(country)) IN ('usa','germany');
```

In [46]:
df_c = df_customers.copy()

res_df = df_c[
    (df_c['country'].str.lower().str.strip().isin(['usa','germany']))
]

res_df = res_df[['country']]

In [45]:
pl_c = pl_customers

res_pl = pl_c.filter(
    (pl.col('country').str.strip_chars().str.to_lowercase().is_in(['usa','germany']))
).select(
    pl.col('country')
)

# BETWEEN

```SQL
SELECT
    score
FROM sales.customers
WHERE score BETWEEN 100 AND 500;
```

In [53]:
df_c = df_customers.copy()

res_df = df_c[
    (df_c['score'].between(100,500))
]

res_df = res_df[['score']]

In [52]:
pl_c = pl_customers

res_pl = pl_c.filter(
    (pl.col('score').is_between(100,500))
).select(
    pl.col('score')
)

# BETWEEN OR NULL

```SQL
SELECT
    score
FROM sales.customers
WHERE (score BETWEEN 100 AND 500) OR (score IS NULL);
```

In [60]:
df_c = df_customers.copy()

res_df = df_c[
    ((df_c['score'].between(100,500)) |
    (df_c['score'].isna())
    )
]

res_df = res_df[['score']]

In [61]:
pl_c = pl_customers

res_pl = pl_c.filter(
    ((pl.col('score').is_between(100,500)) |
    (pl.col('score').is_null()))
).select(
    pl.col('score')
)

# CASE

```SQL
SELECT
    employeeid,
    firstname,
    salary,
    CASE
        WHEN salary > 70000 THEN 'Alto'
        WHEN salary BETWEEN 50000 AND 70000 THEN 'Medio'
        ELSE 'Bajo'
    END AS nivel_salario
FROM sales.employees;
```

In [68]:
df_e = df_employees.copy()

condiciones = [
    (df_e['salary'] > 70000),
    (df_e['salary'].between(50000,70000))
]

elecciones = ['Alto','Medio']

df_e['nivel_salario'] = np.select(condiciones, elecciones, default='Bajo')

df_e = df_e[['employeeid','firstname','salary','nivel_salario']]

In [69]:
pl_e = pl_employees

res_pl = pl_e.with_columns(
    pl.when(pl.col('salary')>70000).then(pl.lit('Alto'))
    .when(pl.col('salary').is_between(50000,70000)).then(pl.lit('Medio'))
    .otherwise(pl.lit('Bajo'))
    .alias('nivel_salario')
).select(
    pl.col('employeeid'),
    pl.col('firstname'),
    pl.col('salary'),
    pl.col('nivel_salario')
)

# COUNT + GROUP BY

```SQL
SELECT
    category,
    COUNT(*)
FROM sales.products
GROUP BY category;
```

In [ ]:
df_p = df_products.copy()

res = df_p.groupby('category').agg(
    total=('product','count')
).reset_index()

In [70]:
pl_p = pl_products

res = pl_p.group_by('category').agg(
    pl.len().alias('total')
)

# GROUP BY / HAVING / COUNT SUM AVG

```SQL
SELECT
    category,
    COUNT(product) AS Cantidad_Productos,
    AVG(price) AS Precio_Promedio,
    SUM(price) AS Valor_Total
FROM sales.products
GROUP BY category
HAVING SUM(price) > 10;
```

In [ ]:
df_p = df_products.copy()

res_df = df_p.groupby('category').agg(
    Cantidad_Productos = ('product','count'),
    Precio_Promedio = ('price', 'mean'),
    Valor_Total = ('price', 'sum')
).reset_index()

res_df = res_df[
    (res_df['Valor_Total'] > 10)
]

In [ ]:
pl_p = pl_products

res_pl = pl_p.group_by('category').agg([
    pl.len().alias('Cantidad_Productos'),
    pl.col('price').mean().alias('Precio_Promedio'),
    pl.col('price').sum().alias('Valor_Total')
]).filter(
    (pl.col('Valor_Total') > 10)
)

# GROUP BY / HAVING / COUNT DISTINCT 

```SQL
SELECT
    orderstatus,
    COUNT(*) AS Total_Pedidos,
    COUNT(DISTINCT customerid) AS Clientes_Unicos
FROM sales.orders
GROUP BY orderstatus
HAVING COUNT(*) > 1;
```

In [ ]:
df_o = df_orders.copy()

res_df = df_o.groupby('orderstatus').agg(
    Total_Pedidos = ('orderid', 'count'),
    Clientes_Unicos = ('customerid', 'nunique')
).reset_index()

res_df = res_df[res_df['Total_Pedidos'] > 1]

In [71]:
pl_o = pl_orders

res_pl = pl_o.group_by('orderstatus').agg([
    pl.col('orderid').count().alias('Total_Pedidos'),
    pl.col('customerid').n_unique().alias('Clientes_Unicos')
]).filter(
    (pl.col('Total_Pedidos') > 1)
)

# INNER JOIN

```SQL
SELECT
    o.orderid,
    c.firstname
FROM sales.orders AS o
INNER JOIN sales.customers AS c ON c.customerid = o.customerid;
```

In [72]:
df_o = df_orders.copy()
df_c = df_customers.copy()

res_df = df_o.merge(
    df_c,
    on='customerid',
    how='inner'
)[['orderid','firstname']]

In [73]:
pl_c = pl_customers
pl_o = pl_orders

res_pl = pl_o.join(
    pl_c,
    on='customerid',
    how='inner'
).select([
    pl.col('orderid'),
    pl.col('firstname')
])

# LEFT JOIN

```SQL
SELECT
    c.firstname,
    o.orderid
FROM sales.customers AS c
LEFT JOIN sales.orders AS o ON c.customerid = o.customerid;
```

In [74]:
df_c = df_customers.copy()
df_o = df_orders.copy()

df_merge = df_c.merge(
    df_o,
    on='customerid',
    how='left'
)[['firstname','orderid']]

In [75]:
pl_c = pl_customers
pl_o = pl_orders

res_pl = pl_c.join(
    pl_o,
    on='customerid',
    how='left'
).select(
    pl.col('firstname'),
    pl.col('orderid')
)

# FULL OUTER JOIN

```SQL
SELECT
    c.lastname,
    o.orderid
FROM sales.customers AS c
FULL OUTER JOIN sales.orders AS o
    ON c.customerid = o.customerid;
```

In [77]:
res_df = df_customers.merge(df_orders, on='customerid', how='outer')

In [79]:
res_pl = pl_customers.join(pl_orders, on='customerid', how='full')

# MULTI JOIN

```SQL
SELECT
    o.orderid,
    o.sales,
    c.firstname AS CustomerFirstNmae,
    c.lastname AS CustomerLastName,
    p.product AS ProductName,
    p.price,
    e.firstname AS EmployeeFirstName,
    e.lastname AS EmployeeLastName
FROM sales.orders AS o
LEFT JOIN sales.customers AS c ON c.customerid = o.customerid
LEFT JOIN sales.products AS p ON p.productid = o.productid
LEFT JOIN sales.employees AS e on e.employeeid = o.salespersonid;
```

In [80]:
df_o = df_orders.copy()
df_c = df_customers.copy()
df_p = df_products.copy()
df_e = df_employees.copy()

df_final = (
    df_o.merge(df_c, on='customerid',how='left')
        .merge(df_p, on='productid', how='left')
        .merge(df_e, left_on='salespersonid', right_on='employeeid', how='left')
)

res_pandas = df_final[[
    'orderid',
    'sales',
    'firstname_x',
    'lastname_x',
    'product',
    'price',
    'firstname_y',
    'lastname_y'
]].rename(columns={
    'firstname_x': 'CustomerFirstName',
    'lastname_x': 'CustomerLastname',
    'firstname_y': 'EmployeeFirstName',
    'lastname_y': 'EmployeeLastName',
    'product': 'ProductName'
})

In [81]:
pl_o = pl_orders
pl_c = pl_customers
pl_p = pl_products
pl_e = pl_employees

pl_join = (
    pl_o.join(pl_c, on='customerid', how='left')
        .join(pl_p, on='productid', how='left')
        .join(pl_e, left_on='salespersonid', right_on='employeeid', how='left')
).select([
    pl.col('orderid'),
    pl.col('sales'),
    pl.col('firstname').alias('CustomerFirstName'),
    pl.col('lastname').alias('CustomerLastName'),
    pl.col('product').alias('ProductName'),
    pl.col('price'),
    pl.col('firstname_right').alias('EmployeeFirstName'),
    pl.col('lastname_right').alias('EmployeeLastname')
])
